<a href="https://colab.research.google.com/github/Akspaks007/DL-Lab/blob/main/Exp_10/Exp_10_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math
import os

In [ ]:
path = "/content/t8.shakespeare.txt"

print('Using dataset:', path)

with open(path, 'r', encoding='utf-8') as f:
    text = f.read()

Using dataset: /content/t8.shakespeare.txt


In [ ]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])
data=torch.tensor(encode(text),dtype=torch.long)

In [ ]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [ ]:
block_size=64
batch_size=32
def get_batch(split):
    d=train_data if split=='train' else val_data
    ix=torch.randint(len(d)-block_size,(batch_size,))
    x=torch.stack([d[i:i+block_size] for i in ix])
    y=torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
n_embd=128
n_heads=4

In [ ]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embd,head_size)
        self.query=nn.Linear(n_embd,head_size)
        self.value=nn.Linear(n_embd,head_size)
        self.tril=torch.tril(torch.ones(block_size,block_size))
    def forward(self,x):
        B,T,C=x.shape
        k=self.key(x); q=self.query(x)
        wei=q@k.transpose(-2,-1)/math.sqrt(C)
        wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        wei=torch.softmax(wei,dim=-1)
        v=self.value(x)
        return wei@v

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads=nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj=nn.Linear(n_embd,n_embd)
    def forward(self,x):
        return self.proj(torch.cat([h(x) for h in self.heads],dim=-1))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_embd,4*n_embd),nn.ReLU(),nn.Linear(4*n_embd,n_embd))
    def forward(self,x): return self.net(x)

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa=MultiHeadAttention(n_heads,n_embd//n_heads)
        self.ffwd=FeedForward()
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)
    def forward(self,x):
        x=x+self.sa(self.ln1(x))
        x=x+self.ffwd(self.ln2(x))
        return x

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa=MultiHeadAttention(n_heads,n_embd//n_heads)
        self.ffwd=FeedForward()
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)
    def forward(self,x):
        x=x+self.sa(self.ln1(x))
        x=x+self.ffwd(self.ln2(x))
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb=nn.Embedding(vocab_size,n_embd)
        self.pos_emb=nn.Embedding(block_size,n_embd)
        self.encoder=nn.Sequential(*[EncoderBlock() for _ in range(2)])
        self.decoder=nn.Sequential(*[DecoderBlock() for _ in range(2)])
        self.ln=nn.LayerNorm(n_embd)
        self.fc=nn.Linear(n_embd,vocab_size)
    def forward(self,x,targets=None):
        B,T=x.shape
        tok=self.token_emb(x)
        pos=self.pos_emb(torch.arange(T))
        x=tok+pos
        x=self.encoder(x)
        x=self.decoder(x)
        logits=self.fc(self.ln(x))
        loss=None
        if targets is not None:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=nn.CrossEntropyLoss()(logits,targets)
        return logits,loss

In [ ]:
model=Transformer()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
for step in range(2000):
    xb,yb=get_batch('train')
    logits,loss=model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step%200==0: print(step,loss.item())

0 4.609665870666504
200 2.250185966491699
400 2.137486696243286
600 1.898754358291626
800 1.8424906730651855
1000 1.630415678024292
1200 1.6435003280639648
1400 1.522918701171875
1600 1.5554559230804443
1800 1.5062105655670166


In [ ]:
def generate(model,idx,max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond=idx[:,-block_size:]
        logits,_=model(idx_cond)
        logits=logits[:,-1,:]
        probs=torch.softmax(logits,dim=-1)
        idx_next=torch.multinomial(probs,1)
        idx=torch.cat((idx,idx_next),dim=1)
    return idx
context=torch.zeros((1,1),dtype=torch.long)
print(decode(generate(model,context,200)[0].tolist()))


    Of thill shouldst that cicking, with be
    Brook you wring, both man, and luce man on I weelay! Are the mad!
                                                                Exits.




<<THUCHIO. 
